In [2]:
import pandas as pd
import numpy as np
from pandas.api.types import ( is_numeric_dtype, is_categorical_dtype, is_object_dtype, is_datetime64_any_dtype )
from default_risk.scripts.cv_mlfow_integration import run_cv_tracked_mlflow
from default_risk.scripts.auxiliar_eda_function import recreate_and_sort_series_given_rows
from default_risk.scripts.auxiliar_eda_function import recreate_and_sort_the_serie_given_ids
import default_risk.config as cfg
from dotenv import load_dotenv
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import mlflow
import mlflow.xgboost
import dtale


bureau_balance_df = pd.read_parquet(cfg.CLEANS_DIR / "bureau_balance.train-cleaned.parquet")

column_order_reference= "months_balance"

bureau_balance_df.sort_values(["id_bureau",column_order_reference],inplace=True)

#aux_function

def get_full_sorted_serie_rows(rows : pd.DataFrame) -> pd.DataFrame:
   return recreate_and_sort_series_given_rows(rows,bureau_balance_df, "id_bureau",column_order_reference)

def get_full_sorted_serie_ids(ids : list) -> pd.DataFrame:
   return recreate_and_sort_the_serie_given_ids(ids,bureau_balance_df, "id_bureau" ,column_order_reference)

def get_time_window_agg_bureau_balance(time_window_in_days: int, df: pd.DataFrame) : 
   time_window_df = df [df["months_balance"] > time_window_in_days] 
   time_window_agg_metrics= time_window_df.groupby("id_curr").agg({
      "status_score": ["max","mean","std"],
      #categoricals
      "status_0": ["mean","sum"],
      "is_delincuency": ["mean","sum"], 
   })
   time_window_in_days= time_window_in_days * -1
   time_window_agg_metrics.columns = [f"balance_first_{time_window_in_days}_{col[0]}_{col[1]}"for col in time_window_agg_metrics.columns]
   time_window_agg_metrics["regularity_ratio"] =  time_window_agg_metrics[f"balance_first_{time_window_in_days}_status_0_sum"] / (time_window_agg_metrics[f"balance_first_{time_window_in_days}_is_delincuency_sum"] + 1)
   time_window_agg_metrics= time_window_agg_metrics.reset_index()
   return time_window_agg_metrics




In [14]:
bureau_balance_df["raw_length"] = bureau_balance_df.groupby("id_bureau").transform("size")
next_status = bureau_balance_df.groupby("id_bureau")["status"].shift(-1)

bureau_balance_df["not_last_row_active"] = (bureau_balance_df["status"] != "C") & (next_status != "C")

bureau_balance_df["amount_rows_with_activity"] = bureau_balance_df.groupby("id_bureau")["not_last_row_active"].transform("sum")

bureau_balance_df["amount_rows_with_activity"]= bureau_balance_df["amount_rows_with_activity"] + 1




In [15]:
#for this, we gonna compute the frecuence and the amount with OHE, but to avoid of neglecting the implict order in a ordinal categorical column like this
#we gonna map to numerical values to be able to consider the severity.
bureau_balance_df["status"]= bureau_balance_df["status"].str.lower()
status_dict = {'c': 0, 'x': 0, '0': 0, '1': 1, '2': 2, '3': 3, '4': 4, '5': 5}
bureau_balance_df["status_score"] = bureau_balance_df["status"].map(status_dict)
bureau_balance_df["status"]=bureau_balance_df["status"].astype("category")
bureau_balance_df= pd.get_dummies(bureau_balance_df, columns=["status"])



In [16]:
bureau_balance_df.head()

,id_bureau,months_balance,closing_month,non_closed_loan,potential_on_going_loan,incomplete_sequence,raw_length,not_last_row_active,amount_rows_with_activity,status_score,status_0,status_1,status_2,status_3,status_4,status_5,status_c,status_x
0,5008804,-15,-12.0,False,False,False,16,True,3,0,False,False,False,False,False,False,False,True
1,5008804,-14,-12.0,False,False,False,16,True,3,0,True,False,False,False,False,False,False,False
2,5008804,-13,-12.0,False,False,False,16,False,3,1,False,True,False,False,False,False,False,False
3,5008804,-12,-12.0,False,False,False,16,False,3,0,False,False,False,False,False,False,True,False
4,5008804,-11,-12.0,False,False,False,16,False,3,0,False,False,False,False,False,False,True,False


In [17]:

agg_from_bureau_balance_dict = {
    "raw_length": ["first"], 
    "amount_rows_with_activity": ["first"],
    "potential_on_going_loan": ["first"],
    "incomplete_sequence": ["first"],
    "closing_month": ["first"],

    "months_balance" : ["min","max"],
    "status_score": ["max","mean","std"],
    
    #categoricals
    "status_0": ["mean","sum"],
    "status_1": ["mean","sum"],
    "status_2": ["mean","sum"],
    "status_3": ["mean","sum"],
    "status_4": ["mean","sum"],
    "status_5": ["mean","sum"],
    "status_c": ["mean","sum"],
    "status_x": ["mean","sum"],
}

In [ ]:
aggregated_bureau_balance= bureau_balance_df.groupby("id_bureau").agg(agg_from_bureau_balance_dict)
aggregated_bureau_balance.columns = [
    f"balance_{col[0]}" if col[1] == "first" else f"balance_{col[0]}_{col[1]}"
    for col in aggregated_bureau_balance.columns
]
agg_metrics_df= aggregated_bureau_balance.reset_index()


In [19]:
aggregated_bureau_balance.to_parquet(cfg.PROCESSED_DIR / "bureu_balance_agg.parquet")

In [20]:
bureau_balance_df = pd.read_parquet(cfg.CLEANS_DIR / "bureau_balance.train-cleaned.parquet")

bureau_balance_df["status"]= bureau_balance_df["status"].str.lower()
status_dict = {'c': 0, 'x': 0, '0': 0, '1': 1, '2': 2, '3': 3, '4': 4, '5': 5}
bureau_balance_df["status_score"] = bureau_balance_df["status"].map(status_dict)
bureau_balance_df["status"]=bureau_balance_df["status"].astype("category")
due_all = bureau_balance_df[bureau_balance_df["status_score"].isin([1, 2, 3, 4, 5])]

most_recents_months_with_dues= due_all.groupby(["id_bureau", "status_score"])["months_balance"].min().unstack()

most_recents_months_with_dues.columns = [f"months_since_{int(col)}_status" for col in most_recents_months_with_dues.columns]
recency_matrix = most_recents_months_with_dues.reset_index()

bureau_balance_df= bureau_balance_df.merge(recency_matrix,how="left",on="id_bureau")

dtale.show(get_full_sorted_serie_rows(bureau_balance_df[bureau_balance_df["months_since_5_status"].notna()]))

In [4]:
bureau_balance_df = pd.read_parquet(cfg.CLEANS_DIR / "bureau_balance.train-cleaned.parquet")

column_order_reference= "months_balance"

bureau_balance_df.sort_values(["id_bureau",column_order_reference],inplace=True)

#aux_function

def get_full_sorted_serie_rows(rows : pd.DataFrame) -> pd.DataFrame:
   return recreate_and_sort_series_given_rows(rows,bureau_balance_df, "id_bureau",column_order_reference)

def get_full_sorted_serie_ids(ids : list) -> pd.DataFrame:
   return recreate_and_sort_the_serie_given_ids(ids,bureau_balance_df, "id_bureau" ,column_order_reference)






bureau_balance_df["raw_length"] = bureau_balance_df.groupby("id_bureau").transform("size")
next_status = bureau_balance_df.groupby("id_bureau")["status"].shift(-1)

bureau_balance_df["not_last_row_active"] = (bureau_balance_df["status"] != "C") & (next_status != "C")

bureau_balance_df["amount_rows_with_activity"] = bureau_balance_df.groupby("id_bureau")["not_last_row_active"].transform("sum")

bureau_balance_df["amount_rows_with_activity"]= bureau_balance_df["amount_rows_with_activity"] + 1

#for this, we gonna compute the frecuence and the amount with OHE, but to avoid of neglecting the implict order in a ordinal categorical column like this
#we gonna map to numerical values to be able to consider the severity.
bureau_balance_df["status"]= bureau_balance_df["status"].str.lower()
status_dict = {'c': 0, 'x': 0, '0': 0, '1': 1, '2': 2, '3': 3, '4': 4, '5': 5}
bureau_balance_df["status_score"] = bureau_balance_df["status"].map(status_dict)
bureau_balance_df["status"]=bureau_balance_df["status"].astype("category")
bureau_balance_df=pd.get_dummies(bureau_balance_df,columns=["status"])

due_all = bureau_balance_df[bureau_balance_df["status_score"].isin([1, 2, 3, 4, 5])]

most_recents_months_with_dues= due_all.groupby(["id_bureau", "status_score"])["months_balance"].max().unstack()



most_recents_months_with_dues.columns = [f"months_since_{int(col)}_status" for col in most_recents_months_with_dues.columns]
recency_matrix = most_recents_months_with_dues.reset_index()

bureau_balance_df= bureau_balance_df.merge(recency_matrix,how="left",on="id_bureau")

bureau_balance_df["is_delincuency"] = bureau_balance_df["status_score"] > 0

bureau_severe= bureau_balance_df [bureau_balance_df["is_delincuency"]]

months= bureau_severe.groupby("id_bureau")["months_balance"].transform("max")

bureau_balance_df["months_since_delincuency"] = bureau_balance_df["id_bureau"].map(months)




agg_from_bureau_balance_dict = {
    "raw_length": ["first"], 
    "amount_rows_with_activity": ["first"],
    "potential_on_going_loan": ["first"],
   # "incomplete_sequence": ["first"],
    "closing_month": ["first"],
    "months_since_delincuency" : ["first"],
    #"month_since_severe_delincuency" : ["first"],
    #"months_since_2_status" : ["first"], 
    #"months_since_3_status" : ["first"], 
    #"months_since_4_status" : ["first"], 
    #"months_since_5_status" : ["first"], 
    "months_balance" : ["min","max"],
    "status_score": ["max","mean","std"],
    #categoricals
    "status_0": ["mean","sum"],
    "is_delincuency": ["mean","sum"], 
}

aggregated_bureau_balance= bureau_balance_df.groupby("id_bureau").agg(agg_from_bureau_balance_dict)
aggregated_bureau_balance.columns = [
    f"balance_{col[0]}" if col[1] == "first" else f"balance_{col[0]}_{col[1]}"
    for col in aggregated_bureau_balance.columns
]
agg_metrics_df= aggregated_bureau_balance.reset_index()

last_90_df =get_time_window_agg_bureau_balance(-90,bureau_balance_df)
last_year_df =get_time_window_agg_bureau_balance(-365,bureau_balance_df)
time_window= last_year_df.merge(last_90_df,how="left",on="id_curr")
time_window["regularity_trend"] = np.where(time_window["balance_first_90_status_0_mean"]!=0, time_window["balance_first_365_status_0_mean"] / time_window["balance_first_90_status_0_mean"],np.nan)
time_window["delincuency_trend"] = np.where(time_window["balance_first_90_is_delincuency_mean"]!=0, time_window["balance_first_365_is_delincuency_mean"] / time_window["balance_first_90_is_delincuency_mean"],np.nan)
time_window["score_trend"] = np.where(time_window["balance_first_90_status_score_mean"]!=0, time_window["balance_first_365_status_score_mean"] / time_window["balance_first_90_status_score_mean"],np.nan)
time_window_to_save= pd.DataFrame()
time_window_to_save["balance_first_90_status_score_mean"] = time_window["balance_first_90_status_score_mean"]
#time_window_to_save["balance_first_365_is_delincuency_mean"] = time_window["balance_first_365_is_delincuency_mean"]
#time_window_to_save["balance_first_90_is_delincuency_mean"] = time_window["balance_first_90_is_delincuency_mean"]
#time_window_to_save["balance_first_365_status_score_mean"] = time_window["balance_first_365_status_score_mean"]
#time_window_to_save["balance_first_365_is_delincuency_sum"] = time_window["balance_first_365_is_delincuency_sum"]
#time_window_to_save["balance_first_90_status_0_mean"] = time_window["balance_first_90_status_0_mean"]
#time_window_to_save["id_curr"] = time_window["id_curr"]

time_window_to_save.to_parquet(cfg.PROCESSED_DIR / "bureau_balance_time_window.parquet")


agg_metrics_df.to_parquet(cfg.PROCESSED_DIR / "bureu_balance_agg.parquet")



In [3]:
bureau_balance_df.head()

,id_bureau,months_balance,closing_month,non_closed_loan,potential_on_going_loan,incomplete_sequence,raw_length,not_last_row_active,amount_rows_with_activity,status_score,...,status_5,status_c,status_x,months_since_1_status,months_since_2_status,months_since_3_status,months_since_4_status,months_since_5_status,is_delincuency,months_since_delincuency
0,5008804,-15,-12.0,False,False,False,16,True,3,0,...,False,False,True,-13.0,NaN,NaN,NaN,NaN,False,NaN
1,5008804,-14,-12.0,False,False,False,16,True,3,0,...,False,False,False,-13.0,NaN,NaN,NaN,NaN,False,NaN
2,5008804,-13,-12.0,False,False,False,16,False,3,1,...,False,False,False,-13.0,NaN,NaN,NaN,NaN,True,NaN
3,5008804,-12,-12.0,False,False,False,16,False,3,0,...,False,True,False,-13.0,NaN,NaN,NaN,NaN,False,NaN
4,5008804,-11,-12.0,False,False,False,16,False,3,0,...,False,True,False,-13.0,NaN,NaN,NaN,NaN,False,NaN
